In [ ]:
# This notebook will produce a feature table with mapped MS2 spectra and export the selected MS2 spectra in MSP format. 
# This will allow your data to be analyzed using GNPS or other MS2-based tools.

In [ ]:
# Library imports for data processing

import pymzml
import os
import tqdm
import pandas as pd
import matchms
import intervaltree

In [ ]:
# Set paths for dataset and find all MS2 spectra

DATASET_DIR = "/Users/mitchjo/Downloads/GNPS/"
all_mzml_files = []
for f in os.listdir(DATASET_DIR):
    if f.lower().endswith("mzml"):
        all_mzml_files.append(os.path.join(DATASET_DIR, f))

spectra = []
for mzml in tqdm.tqdm(all_mzml_files):
    for spec in pymzml.run.Reader(mzml):
        if spec.ms_level == 2:
            spec_rtime = spec.scan_time_in_minutes()*60
            for precursor in spec.selected_precursors:
                entry = {"spectrum": spec,
                         "rtime": spec_rtime,
                         "sample_origin": os.path.basename(mzml).rstrip(".mzML"),
                         "intensity_sum": sum(spec.i)}
                entry.update(precursor)
                spectra.append(entry)

In [ ]:
# read the feature table from the Asari run


FEATURE_TABLE_PATH = "/Users/mitchjo/Downloads/GNPS/output_asari_project_117114923/preferred_Feature_table.tsv"

ft = pd.read_csv(FEATURE_TABLE_PATH, sep="\t")

samples = ft.columns[11:]
non_sample = ft.columns[:11]
features = [f for f in tqdm.tqdm(ft.to_dict(orient='records'))]


In [ ]:
def map_features_to_ms2(
    features,
    ms2_spectra,
    ppm_tol: float = 5.0,
    rt_tol: float = 10.0,
) -> pd.DataFrame:
    """
    Map each MS1 feature to its most likely MS2 spectrum based on m/z and RT, enforcing:
      - one feature -> at most one MS2 (unique MS2 usage)
      - 5 ppm m/z tolerance (primary)
      - 10 s RT tolerance (secondary)
      - disallow mapping if feature intensity == 0 in ms2's origin sample.
      - choose minimal combined normalized deviation:
            sqrt[(Δppm/ppm_tol)^2 + (Δrt/rt_tol)^2]
    Returns DataFrame with columns: feature_id, ms2_id (or None).
    """
    spectrum_mz_tree = intervaltree.IntervalTree()
    spectrum_rt_tree = intervaltree.IntervalTree()
    id_to_spectrum = {}
    for spectrum in ms2_spectra:
        mz_err = spectrum['mz']/1e6 * ppm_tol
        id_to_spectrum[(spectrum['sample_origin'], spectrum['spectrum'].ID)] = spectrum
        spectrum_mz_tree.addi(spectrum['mz'] - mz_err, spectrum['mz'] + mz_err, (spectrum['sample_origin'], spectrum['spectrum'].ID))
        spectrum_rt_tree.addi(spectrum['rtime'] - rt_tol, spectrum['rtime'] + rt_tol, (spectrum['sample_origin'], spectrum['spectrum'].ID))
    update_features = []
    for feature in features:
        feature['matches'] = set()
        mz_matches = {x.data for x in spectrum_mz_tree.at(feature['mz'])}
        if mz_matches:
            rt_matches = {x.data for x in spectrum_rt_tree.at(feature['rtime'])}
            if rt_matches:
                intersection = mz_matches.intersection(rt_matches)
                feature['matches'] = intersection
        update_features.append(feature)
    return update_features, id_to_spectrum

mapped_features, id_to_spectrum = map_features_to_ms2(features, spectra)

features_w_ms2 = []
for feature in [f for f in mapped_features if f['matches']]:
    feature['selected_ms2'] = ''
    possibles = []
    for spec_key in feature['matches']:
        if feature[spec_key[0]] > 0:
            possibles.append((spec_key[0], spec_key[1], feature[spec_key[0]]))
    if possibles:
        spectra_w_errors = []
        for possible in possibles:
            spectrum_obj = id_to_spectrum[(possible[0], possible[1])]
            rtime_err = abs(spectrum_obj['rtime'] - feature['rtime'])
            mz_err = abs(spectrum_obj['mz'] - feature['mz'])
            spectra_w_errors.append((rtime_err, mz_err, spectrum_obj, round(spectrum_obj['mz'], 4), round(spectrum_obj['rtime'], 4), possible[0]))
        feature['selected_ms2'] = sorted(spectra_w_errors, key=lambda x: x[1])[0]
    del feature['matches']
    features_w_ms2.append(feature)


spectra_to_export = []
for f in features_w_ms2:
    if f['selected_ms2'] and f['selected_ms2'][2]:
        ms2_spectrum = f['selected_ms2'][2]['spectrum']
        precursor = f['selected_ms2'][3]
        spec_for_export = matchms.Spectrum(
            ms2_spectrum.mz,
            ms2_spectrum.i,
            metadata = {
                "id": f"{f['selected_ms2'][3]}_{f['selected_ms2'][4]}_{f['selected_ms2'][5]}",
                "precursor_mz": f['selected_ms2'][3],
                "observed_rtime": f['selected_ms2'][4],
                "charge": 1
            }
        )
        spec_for_export = matchms.filtering.default_filters(spec_for_export)
        spec_for_export = matchms.filtering.normalize_intensities(spec_for_export)
        f['selected_ms2'] = f"{f['selected_ms2'][3]}_{f['selected_ms2'][4]}_{f['selected_ms2'][5]}"
        spectra_to_export.append(spec_for_export)

os.remove("./extracted_ms2_spectra.msp")
matchms.exporting.save_spectra(spectra_to_export, "extracted_ms2_spectra.msp")
df = pd.DataFrame(features_w_ms2)
new_df = pd.DataFrame()
for x in non_sample:
    new_df[x] = df[x]
new_df['selected_ms2'] = df['selected_ms2']
for x in samples:
    new_df[x] = df[x]
pd.DataFrame(new_df).to_csv('feature_table_w_ms2.tsv', sep="\t", index=False)
